In [ ]:
# 1. MOUNT DRIVE AND SETUP
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import json
import pandas as pd
import numpy as np
import nibabel as nib
from tqdm import tqdm
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

# 2. CONFIGURATION
class Config:
    # Paths
    DATASET_PATH = '/content/drive/MyDrive/MMDental/MMDental'
    SAVE_DIR = '/content/drive/MyDrive/MMDental/models'

    # Data parameters
    TARGET_SIZE = (64, 64, 64)  # Reduced for memory efficiency
    BATCH_SIZE = 2  # Start with 2, increase if GPU memory allows
    NUM_WORKERS = 2

    # Training parameters
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    NUM_EPOCHS = 30
    PATIENCE = 8

    # Random seed
    SEED = 42
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = Config()
os.makedirs(config.SAVE_DIR, exist_ok=True)

def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed(config.SEED)

# 3. LOAD AND PROCESS MEDICAL RECORDS
print("\n" + "="*60)
print("LOADING MEDICAL RECORDS")
print("="*60)

csv_path = os.path.join(config.DATASET_PATH, 'medical_records.csv')
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} records")

# Define dental conditions (8 main categories from the paper)
DENTAL_CONDITIONS = [
    'Malformed teeth',
    'Pulpitis',
    'Impacted tooth',
    'Missing tooth',
    'Dental caries',
    'Periodontitis',
    'Gingivitis',
    'Apical periodontitis'
]

def map_diagnosis(diagnosis):
    if pd.isna(diagnosis):
        return 'Unknown'
    diag = str(diagnosis).lower()

    if any(word in diag for word in ['malformed', 'crooked', 'misaligned']):
        return 'Malformed teeth'
    elif 'pulpitis' in diag:
        return 'Pulpitis'
    elif 'impacted' in diag:
        return 'Impacted tooth'
    elif any(word in diag for word in ['missing', 'loss', 'edentulous']):
        return 'Missing tooth'
    elif 'caries' in diag:
        return 'Dental caries'
    elif 'periodontitis' in diag:
        return 'Periodontitis'
    elif 'gingivitis' in diag:
        return 'Gingivitis'
    elif 'apical' in diag:
        return 'Apical periodontitis'
    else:
        return 'Other'

df['condition'] = df['Diagnosis'].apply(map_diagnosis)
df = df[df['condition'].isin(DENTAL_CONDITIONS)]
print(f"Filtered to {len(df)} records with known conditions")
print("\nCondition distribution:")
print(df['condition'].value_counts())

# 4. MATCH CBCT FILES
print("\n" + "="*60)
print("MATCHING CBCT FILES")
print("="*60)

# Find all patient folders and CBCT files
cbct_files = {}
for folder_name in os.listdir(config.DATASET_PATH):
    folder_path = os.path.join(config.DATASET_PATH, folder_name)
    if os.path.isdir(folder_path) and folder_name.isdigit():
        for file_name in os.listdir(folder_path):
            if file_name.endswith('.nii.gz'):
                cbct_files[folder_name] = os.path.join(folder_path, file_name)
                break

print(f"Found {len(cbct_files)} CBCT files")

# Merge with dataframe
df['patient_id'] = df['Filename'].astype(str)
df['cbct_path'] = df['patient_id'].map(cbct_files)
df_with_cbct = df[df['cbct_path'].notna()].copy()
print(f"Patients with both records and CBCT: {len(df_with_cbct)}")

# 5. CREATE TRAIN/VAL/TEST SPLIT
print("\n" + "="*60)
print("CREATING DATA SPLITS")
print("="*60)

# Get unique patients
unique_patients = df_with_cbct['patient_id'].unique()
print(f"Unique patients: {len(unique_patients)}")

# Stratified split by condition
patient_primary = df_with_cbct.groupby('patient_id')['condition'].first().reset_index()

train_patients, temp_patients = train_test_split(
    patient_primary['patient_id'].values,
    test_size=0.3,
    random_state=config.SEED,
    stratify=patient_primary['condition']
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.5,
    random_state=config.SEED
)

print(f"Train patients: {len(train_patients)}")
print(f"Validation patients: {len(val_patients)}")
print(f"Test patients: {len(test_patients)}")

train_df = df_with_cbct[df_with_cbct['patient_id'].isin(train_patients)]
val_df = df_with_cbct[df_with_cbct['patient_id'].isin(val_patients)]
test_df = df_with_cbct[df_with_cbct['patient_id'].isin(test_patients)]

print(f"\nTrain records: {len(train_df)}")
print(f"Val records: {len(val_df)}")
print(f"Test records: {len(test_df)}")

# 6. CUSTOM DATASET CLASS
class CBCTDataset(Dataset):
    def __init__(self, dataframe, target_size, label_encoder=None, augment=False):
        self.dataframe = dataframe.reset_index(drop=True)
        self.target_size = target_size
        self.augment = augment

        if label_encoder is None:
            self.label_encoder = LabelEncoder()
            self.labels = self.label_encoder.fit_transform(dataframe['condition'])
        else:
            self.label_encoder = label_encoder
            self.labels = self.label_encoder.transform(dataframe['condition'])

    def _load_and_resize(self, path):
        try:
            nifti = nib.load(path)
            volume = nifti.get_fdata().astype(np.float32)

            # Normalize
            volume = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)

            # Resize using simple slicing (faster than scipy)
            from scipy.ndimage import zoom
            factors = (self.target_size[0]/volume.shape[0],
                      self.target_size[1]/volume.shape[1],
                      self.target_size[2]/volume.shape[2])
            volume = zoom(volume, factors, order=1)

            return np.expand_dims(volume, axis=0)
        except Exception as e:
            print(f"Error loading {path}: {e}")
            return np.zeros((1, *self.target_size))

    def _augment(self, volume):
        if not self.augment:
            return volume

        # Random rotations
        if np.random.random() > 0.5:
            k = np.random.randint(1, 4)
            volume = torch.rot90(volume, k, dims=(1, 2))

        # Random flips
        if np.random.random() > 0.5:
            volume = torch.flip(volume, dims=[2])

        # Random noise
        if np.random.random() > 0.7:
            volume = volume + torch.randn_like(volume) * 0.05
            volume = torch.clamp(volume, 0, 1)

        return volume

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        volume = self._load_and_resize(row['cbct_path'])
        volume = torch.from_numpy(volume).float()

        if self.augment:
            volume = self._augment(volume)

        return volume, torch.tensor(self.labels[idx], dtype=torch.long)

# 7. 3D CNN MODEL (Simplified for faster training)
class Simple3DCNN(nn.Module):
    def __init__(self, num_classes=8):
        super(Simple3DCNN, self).__init__()

        self.conv_layers = nn.Sequential(
            # Block 1
            nn.Conv3d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(2),

            # Block 2
            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(2),

            # Block 3
            nn.Conv3d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(2),

            # Block 4
            nn.Conv3d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm3d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool3d((1, 1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# 8. CREATE DATALOADERS
print("\n" + "="*60)
print("CREATING DATALOADERS")
print("="*60)

label_encoder = LabelEncoder()
label_encoder.fit(train_df['condition'])

train_dataset = CBCTDataset(train_df, config.TARGET_SIZE, label_encoder, augment=True)
val_dataset = CBCTDataset(val_df, config.TARGET_SIZE, label_encoder, augment=False)
test_dataset = CBCTDataset(test_df, config.TARGET_SIZE, label_encoder, augment=False)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# Calculate class weights
class_counts = train_df['condition'].value_counts()
class_counts = class_counts.reindex(label_encoder.classes_, fill_value=1)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_counts)
class_weights_tensor = torch.tensor(class_weights.values, dtype=torch.float32).to(config.DEVICE)

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=config.NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS)

# 9. TRAINING SETUP
print("\n" + "="*60)
print("SETTING UP TRAINING")
print("="*60)

model = Simple3DCNN(num_classes=len(label_encoder.classes_)).to(config.DEVICE)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
scaler = GradScaler() if torch.cuda.is_available() else None

# 10. TRAINING LOOP
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)

best_val_acc = 0
best_model_path = os.path.join(config.SAVE_DIR, 'best_model.pth')
patience_counter = 0

train_losses, val_losses, train_accs, val_accs = [], [], [], []

for epoch in range(1, config.NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{config.NUM_EPOCHS}")
    print("-" * 40)

    # Training
    model.train()
    train_loss = 0
    train_preds, train_labels = [], []

    for volumes, labels in tqdm(train_loader, desc="Training"):
        volumes, labels = volumes.to(config.DEVICE), labels.to(config.DEVICE)

        optimizer.zero_grad()

        if scaler:
            with autocast():
                outputs = model(volumes)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(volumes)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

    train_loss /= len(train_loader)
    train_acc = accuracy_score(train_labels, train_preds)

    # Validation
    model.eval()
    val_loss = 0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for volumes, labels in tqdm(val_loader, desc="Validation"):
            volumes, labels = volumes.to(config.DEVICE), labels.to(config.DEVICE)

            if scaler:
                with autocast():
                    outputs = model(volumes)
                    loss = criterion(outputs, labels)
            else:
                outputs = model(volumes)
                loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_loss /= len(val_loader)
    val_acc = accuracy_score(val_labels, val_preds)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"LR: {optimizer.param_groups[0]['lr']:.6f}")

    # Learning rate scheduling
    scheduler.step(val_loss)

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'model_state_dict': model.state_dict(),
            'label_encoder': label_encoder,
            'class_names': label_encoder.classes_.tolist(),
            'val_acc': val_acc
        }, best_model_path)
        patience_counter = 0
        print(f"✓ Best model saved (val_acc: {val_acc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= config.PATIENCE:
            print(f"Early stopping after {epoch} epochs")
            break

# 11. EVALUATION ON TEST SET

print("\n" + "="*60)
print("EVALUATING ON TEST SET")
print("="*60)

# Load the saved checkpoint with weights_only=False (safe since you saved it)
try:
    checkpoint = torch.load(best_model_path, map_location=config.DEVICE, weights_only=False)
    print("✅ Model loaded successfully")
except Exception as e:
    print(f"Error loading: {e}")
    # Fallback try with different method
    checkpoint = torch.load(best_model_path, map_location=config.DEVICE, weights_only=False, pickle_module=pickle)

# Load model weights
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Get the label encoder (either from checkpoint or recreate from train_df)
if 'label_encoder' in checkpoint:
    label_encoder = checkpoint['label_encoder']
    print(f"Loaded label encoder with classes: {label_encoder.classes_.tolist()}")
else:
    # Fallback: recreate label encoder
    label_encoder = LabelEncoder()
    label_encoder.fit(train_df['condition'])
    print("Recreated label encoder from training data")

# Evaluate on test set
test_preds, test_labels = [], []
test_loss = 0

with torch.no_grad():
    for volumes, labels in tqdm(test_loader, desc="Testing"):
        volumes = volumes.to(config.DEVICE)
        labels_np = labels.numpy()

        outputs = model(volumes)
        _, preds = torch.max(outputs, 1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels_np)

# Calculate metrics
test_acc = accuracy_score(test_labels, test_preds)
test_f1 = f1_score(test_labels, test_preds, average='weighted')
test_precision = precision_score(test_labels, test_preds, average='weighted', zero_division=0)
test_recall = recall_score(test_labels, test_preds, average='weighted')

print(f"\n📊 Test Results:")
print(f"  Accuracy:  {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"  F1-Score:  {test_f1:.4f}")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall:    {test_recall:.4f}")

print("\n📋 Detailed Classification Report:")
print(classification_report(test_labels, test_preds,
                           target_names=label_encoder.classes_,
                           digits=4))

# Confusion Matrix
print("\n📊 Confusion Matrix:")
cm = confusion_matrix(test_labels, test_preds)
print(cm)

# Optional: Visualize confusion matrix
try:
    import matplotlib.pyplot as plt
    import seaborn as sns

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_)
    plt.title('Confusion Matrix - Test Set')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()

    cm_path = os.path.join(config.SAVE_DIR, 'confusion_matrix.png')
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Confusion matrix saved to: {cm_path}")
except Exception as e:
    print(f"Could not plot confusion matrix: {e}")

# ============================================
# SAVE FINAL MODEL (Fixed version)
# ============================================

print("\n" + "="*60)
print("SAVING FINAL MODEL")
print("="*60)

# Store model state only (separate from encoder) for easier loading
final_model_path = os.path.join(config.SAVE_DIR, 'final_model_weights_only.pth')
torch.save(model.state_dict(), final_model_path)
print(f"✅ Model weights saved to: {final_model_path}")

# Also save full checkpoint (for compatibility)
full_checkpoint_path = os.path.join(config.SAVE_DIR, 'full_checkpoint.pth')
torch.save({
    'model_state_dict': model.state_dict(),
    'label_encoder_classes': label_encoder.classes_.tolist(),  # Save just the classes, not the encoder object
    'config': {
        'target_size': config.TARGET_SIZE,
        'num_classes': len(label_encoder.classes_)
    },
    'test_metrics': {
        'accuracy': test_acc,
        'f1_score': test_f1,
        'precision': test_precision,
        'recall': test_recall
    }
}, full_checkpoint_path)
print(f"✅ Full checkpoint saved to: {full_checkpoint_path}")

# ============================================
# IMPROVED INFERENCE FUNCTION (Works with both loading methods)
# ============================================

def predict_cbct(cbct_path, model, class_names, device, target_size=(64,64,64)):
    """
    Predict dental condition from CBCT file

    Args:
        cbct_path: Path to .nii.gz file
        model: Trained PyTorch model
        class_names: List of class names (strings)
        device: 'cuda' or 'cpu'
        target_size: (depth, height, width)

    Returns:
        Dictionary with predictions
    """
    model.eval()

    # Load and preprocess
    nifti = nib.load(cbct_path)
    volume = nifti.get_fdata().astype(np.float32)
    volume = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)

    # Resize
    from scipy.ndimage import zoom
    factors = (target_size[0]/volume.shape[0],
              target_size[1]/volume.shape[1],
              target_size[2]/volume.shape[2])
    volume = zoom(volume, factors, order=1)
    volume = np.expand_dims(volume, axis=(0, 1))
    volume = torch.from_numpy(volume).float().to(device)

    # Predict
    with torch.no_grad():
        outputs = model(volume)
        probs = torch.nn.functional.softmax(outputs, dim=1)

    probs = probs.cpu().numpy()[0]
    pred_idx = np.argmax(probs)
    predicted_condition = class_names[pred_idx]
    confidence = float(probs[pred_idx])

    # Get top 3 predictions
    top3_idx = np.argsort(probs)[-3:][::-1]
    top3 = [(class_names[i], float(probs[i])) for i in top3_idx]

    return {
        'predicted_condition': predicted_condition,
        'confidence': confidence,
        'top3_predictions': top3,
        'all_probabilities': {class_names[i]: float(p) for i, p in enumerate(probs)}
    }

# ============================================
# LOAD MODEL FOR INFERENCE (Two ways)
# ============================================

print("\n" + "="*60)
print("LOADING MODEL FOR INFERENCE")
print("="*60)

# Way 1: Load just weights (simpler, no sklearn dependency)
def load_model_for_inference(weights_path, class_names, device='cpu'):
    """Load model from weights only"""
    model = Simple3DCNN(num_classes=len(class_names))
    model.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
    model.to(device)
    model.eval()
    return model

# Way 2: Load from full checkpoint
def load_model_from_checkpoint(checkpoint_path, device='cpu'):
    """Load model from full checkpoint"""
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model = Simple3DCNN(num_classes=len(checkpoint['label_encoder_classes']))
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    return model, checkpoint['label_encoder_classes']

# Example: Load using weights-only (safer)
class_names = label_encoder.classes_.tolist()
inference_model = load_model_for_inference(final_model_path, class_names, config.DEVICE)
print(f"✅ Model loaded for inference with classes: {class_names}")

# ============================================
# TEST INFERENCE ON A SAMPLE
# ============================================

print("\n" + "="*60)
print("TESTING INFERENCE ON A SAMPLE")
print("="*60)

# Get a test sample
if len(test_df) > 0:
    sample_row = test_df.iloc[0]
    sample_path = sample_row['cbct_path']
    true_label = sample_row['condition']

    print(f"\n📁 Testing on patient: {sample_row['patient_id']}")
    print(f"📝 True condition: {true_label}")

    result = predict_cbct(sample_path, inference_model, class_names, config.DEVICE)

    print(f"\n🔮 Prediction: {result['predicted_condition']}")
    print(f"📊 Confidence: {result['confidence']:.2%}")

    print("\n🎯 Top 3 predictions:")
    for i, (cond, conf) in enumerate(result['top3_predictions'], 1):
        print(f"   {i}. {cond}: {conf:.2%}")

print("\n" + "="*60)
print("✅ COMPLETE! Model saved and ready for use")
print("="*60)


# Get a test sample
test_row = test_df.iloc[0]
result = predict_cbct(test_row['cbct_path'], model, label_encoder, config.DEVICE)

print(f"\nPatient ID: {test_row['patient_id']}")
print(f"True condition: {test_row['condition']}")
print(f"Predicted: {result['predicted_condition']} (confidence: {result['confidence']:.4f})")
print("\nTop 3 predictions:")
for cond, conf in result['top3_predictions']:
    print(f"  {cond}: {conf:.4f}")

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)